# SciQLop Histogram2D

`Histogram2D` bins scatter data into a 2D density grid rendered as a color map.
Useful for phase-space distributions, correlation plots, or any (x, y) scatter.

<div align="center">
<img src="https://github.com/SciQLop/SciQLop/raw/main/SciQLop/resources/icons/SciQLop.png"/>
</div>

## 1. Creating a histogram from scatter data

In [ ]:
import numpy as np
from SciQLop.user_api.plot import create_plot_panel

# Generate sample scatter data: a noisy ring
rng = np.random.default_rng(42)
theta = rng.uniform(0, 2 * np.pi, 50_000)
r = 1.0 + 0.3 * rng.standard_normal(50_000)
x = r * np.cos(theta)
y = r * np.sin(theta)

p = create_plot_panel()
xy_plot, hist = p.histogram2d(x, y, x_bins=80, y_bins=80)

## 2. Logarithmic color scale

When counts span several orders of magnitude, a log scale improves contrast.

In [ ]:
hist.z_log_scale = True

## 3. Updating data

Call `set_data(x, y)` to re-bin with new scatter data without recreating the plot.

In [ ]:
# Two gaussian clusters
x2 = np.concatenate([rng.normal(-1, 0.5, 20_000), rng.normal(1, 0.3, 20_000)])
y2 = np.concatenate([rng.normal(0, 0.5, 20_000), rng.normal(1, 0.3, 20_000)])
hist.set_data(x2, y2)

## 4. Visibility

In [ ]:
hist.visible = False
hist.visible = True

## 5. Adding a histogram to an existing plot

Instead of creating a new plot, you can add a histogram directly onto an existing `XYPlot`.

In [ ]:
hist2 = xy_plot.histogram2d(x2, y2, name="clusters", x_bins=60, y_bins=60)

## 6. Callable histogram (live update on pan/zoom)

Instead of passing static arrays, pass a **callback function**. SciQLop will
call it automatically whenever the time range changes, re-computing the scatter on the fly.

The callback receives `(start, stop)` as epoch floats (same as virtual products)
and should return `(x, y)` arrays.

In [ ]:
import numpy as np
import speasy as spz
from SciQLop.user_api.plot import create_plot_panel
from SciQLop.user_api import TimeRange
from datetime import datetime

def bxy_correlation(start, stop):
    """Fetch MMS1 FGM data and return (Bx, By) for a 2D histogram."""
    b = spz.get_data("cda/MMS1_FGM_SRVY_L2/mms1_fgm_b_gse_srvy_l2", start, stop)
    if b is None:
        return np.array([]), np.array([])
    return b.values[:, 0].astype(np.float64), b.values[:, 1].astype(np.float64)

p3 = create_plot_panel()
p3.time_range = TimeRange(
    datetime(2015, 11, 18, 2, 14, 30),
    datetime(2015, 11, 18, 3, 34, 0)
)
_, live_hist = p3.histogram2d(bxy_correlation, x_bins=80, y_bins=80, z_log_scale=True)

## 7. Real-data example: Bx vs By correlation

Plot the correlation between two magnetic field components using static data.

In [ ]:
import speasy as spz
from datetime import datetime

b_gse = spz.get_data(
    "cda/MMS1_FGM_SRVY_L2/mms1_fgm_b_gse_srvy_l2",
    datetime(2015, 11, 18, 2, 14, 30),
    datetime(2015, 11, 18, 3, 34, 0)
)

if b_gse is not None:
    bx = b_gse.values[:, 0]
    by = b_gse.values[:, 1]

    p4 = create_plot_panel()
    _, corr_hist = p4.histogram2d(bx, by, x_bins=100, y_bins=100, z_log_scale=True)

## API reference

| Method / Property | Description |
|---|---|
| `panel.histogram2d(x, y, *, x_bins, y_bins, z_log_scale)` | Create histogram from static data |
| `panel.histogram2d(callback, *, x_bins, y_bins, z_log_scale)` | Create histogram with live callback |
| `plot.histogram2d(...)` | Add histogram to an existing plot (same signatures) |
| `hist.set_data(x, y)` | Update scatter data (static histograms only) |
| `.z_log_scale` | Toggle log color scale |
| `.visible` | Show/hide the histogram |
| `.gradient` | Color gradient |

**Callable signature:** `f(start, stop) -> (x, y)` where `start` and `stop` are
epoch floats (same convention as virtual products).